# 🫀 Cardiovascular Heart Disease Prediction using Logistic Regression

## 🩺 Problem Statement
Cardiovascular diseases (CVDs) are the **#1 cause of death globally**, taking an estimated 17.9 million lives each year according to the World Health Organization (WHO).
Early identification of individuals at high risk allows for timely medical interventions (lifestyle changes, drug therapies, or catheterizations) before fatal myocardial infarction (heart attack) occurs.

## 🎯 Objective
Build, evaluate, and interpret a **Logistic Regression** clinical classification model to:
1. Accurately predict whether a patient has **Heart Disease (`1`)** or is **Normal (`0`)** based on 11 non-invasive clinical biomarkers.
2. Derive **Clinical Odds Ratios ($e^{\beta}$)** to quantify the relative risk contributed by each biomarker (e.g., ST depression, exercise angina, fasting blood sugar).
3. Perform **Clinical Threshold Tuning** to minimize fatal False Negatives in emergency triage.
4. Rigorously validate the **5 Core Statistical Assumptions of Logistic Regression**.

## 📊 Dataset
**Heart Failure Prediction Dataset** from Kaggle (`fedesoriano/heart-failure-prediction`)
- 918 patient records combining 5 renowned cardiology clinical datasets (Cleveland, Hungarian, Switzerland, Long Beach VA, and Statlog)
- Target Variable: `HeartDisease` (0 = Normal, 1 = Heart Disease)

---
## Step 1: Download Dataset from Kaggle

In [ ]:
import kagglehub

# Download latest version from Kaggle
path = kagglehub.dataset_download("fedesoriano/heart-failure-prediction")
print("Dataset files downloaded to:", path)

---
## Step 2: Import Libraries

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report, log_loss
)
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings('ignore')
print("Libraries successfully imported!")

---
## Step 3: Load & Explore the Dataset

In [ ]:
df = pd.read_csv(os.path.join(path, 'heart.csv'))

print("First 5 Patient Records:")
display(df.head())

print("\nDataset Shape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistical Summary:")
display(df.describe())

print("\nCategorical Attributes & Unique Values:")
for col in df.select_dtypes(include='object').columns:
    print(f"  {col}: {df[col].unique().tolist()}")

---
## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# 1. Target Variable Distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

counts = df['HeartDisease'].value_counts()
axes[0].bar(['Normal (0)', 'Heart Disease (1)'], counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.5)
axes[0].set_title('Heart Disease Patient Count', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Patients')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, f"{v} ({v/len(df)*100:.1f}%)", ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['Heart Disease (1)', 'Normal (0)'], colors=['#e74c3c', '#2ecc71'],
            autopct='%1.1f%%', startangle=140, explode=[0.05, 0], shadow=True)
axes[1].set_title('Target Proportion', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 2. Correlation Heatmap (Numerical Clinical Biomarkers)
plt.figure(figsize=(9, 7))
num_corr = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(num_corr, annot=True, cmap='coolwarm', center=0, fmt='.3f', linewidths=0.5, square=True)
plt.title('Correlation Heatmap: Numerical Biomarkers vs Heart Disease', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Key Categorical Risk Factors Analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.countplot(data=df, x='Sex', hue='HeartDisease', palette='Set2', ax=axes[0, 0])
axes[0, 0].set_title('Heart Disease Prevalence by Biological Sex')

sns.countplot(data=df, x='ChestPainType', hue='HeartDisease', palette='Set2', ax=axes[0, 1])
axes[0, 1].set_title('Heart Disease by Chest Pain Type (ASY = Asymptomatic)')

sns.countplot(data=df, x='ExerciseAngina', hue='HeartDisease', palette='Set2', ax=axes[1, 0])
axes[1, 0].set_title('Heart Disease by Exercise-Induced Angina')

sns.countplot(data=df, x='ST_Slope', hue='HeartDisease', palette='Set2', ax=axes[1, 1])
axes[1, 1].set_title('Heart Disease by ECG ST Slope (Up / Flat / Down)')

plt.tight_layout()
plt.show()

---
## Step 5: Data Preprocessing & Feature Engineering

In [ ]:
# Handle resting cholesterol 0 values (recorded as 0 in Hungarian center) by imputing positive median
chol_median = df[df['Cholesterol'] > 0]['Cholesterol'].median()
df['Cholesterol'] = df['Cholesterol'].replace(0, chol_median)

# One-Hot Encoding for categorical attributes (drop_first=True to avoid dummy variable trap)
df_encoded = pd.get_dummies(df, columns=['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope'], drop_first=True)

# Feature Engineering: Cardiac Ischemia Stress Index
# High ST depression combined with exercise-induced angina represents severe compounding ischemic risk
df_encoded['stress_risk_index'] = df_encoded['Oldpeak'] * df_encoded['ExerciseAngina_Y']

print("Encoded Features List:")
print(df_encoded.columns.tolist())
print("\nShape after Preprocessing & Feature Engineering:", df_encoded.shape)
display(df_encoded.head())

---
## Step 6: Define Features (X) and Target (y)

In [ ]:
X = df_encoded.drop('HeartDisease', axis=1)
y = df_encoded['HeartDisease']

print(f"Feature matrix (X) shape: {X.shape}")
print(f"Target vector (y) shape: {y.shape}")

---
## Step 7: Multicollinearity Diagnosis using VIF

In [ ]:
vif_df = pd.DataFrame()
vif_df['Feature'] = X.columns
vif_df['VIF'] = [variance_inflation_factor(X.astype(float).values, i) for i in range(X.shape[1])]
vif_df = vif_df.sort_values('VIF', ascending=False).reset_index(drop=True)

print("Variance Inflation Factor (VIF) Analysis:")
print("=" * 45)
print(vif_df.to_string(index=False))
print("=" * 45)
print("Rule: VIF < 10 confirms no severe multicollinearity.")

---
## Step 8: Stratified Train-Test Split

In [ ]:
# Stratified split ensures identical disease proportions in train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set samples: {X_train.shape[0]} ({y_train.mean():.2%} positive)")
print(f"Testing set samples:  {X_test.shape[0]} ({y_test.mean():.2%} positive)")

---
## Step 9: Feature Standardization (StandardScaler)

In [ ]:
# StandardScaler normalizes continuous biomarkers so regularized logistic regression weights are directly comparable
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully (mean=0, std=1 on training data).")

---
## Step 10: Train Logistic Regression Model & Calculate Odds Ratios

In [ ]:
# Train Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
model.fit(X_train_scaled, y_train)

print("Logistic Regression Model Trained Successfully!")
print(f"Intercept (β₀): {model.intercept_[0]:.4f}")

# Odds Ratios: OR = exp(beta)
odds_ratios = np.exp(model.coef_[0])
coeff_summary = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient (β)': model.coef_[0],
    'Odds Ratio (e^β)': odds_ratios
}).sort_values('Odds Ratio (e^β)', ascending=False).reset_index(drop=True)

print("\nClinical Odds Ratios Summary:")
display(coeff_summary)

---
## Step 11: Make Predictions & Probability Estimations

In [ ]:
# Binary class predictions (default 0.5 threshold)
y_pred = model.predict(X_test_scaled)
# Calibrated posterior probabilities P(HeartDisease=1 | X)
y_probs = model.predict_proba(X_test_scaled)[:, 1]

results_preview = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred[:10],
    'Probability P(Disease)': np.round(y_probs[:10], 4)
})
print("First 10 Test Predictions vs Actual:")
display(results_preview)

---
## Step 12: Comprehensive Model Evaluation

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_probs)
loss = log_loss(y_test, y_probs)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)

print("=" * 55)
print("   Logistic Regression Clinical Performance Report")
print("=" * 55)
print(f"  Accuracy:            {acc:.4f} ({acc*100:.2f}%)")
print(f"  Sensitivity (Recall): {rec:.4f} ({rec*100:.2f}%)")
print(f"  Specificity:         {specificity:.4f} ({specificity*100:.2f}%)")
print(f"  Precision:           {prec:.4f} ({prec*100:.2f}%)")
print(f"  F1-Score:            {f1:.4f}")
print(f"  ROC-AUC Score:       {roc_auc:.4f}")
print(f"  Cross-Entropy Loss:  {loss:.4f}")
print("=" * 55)
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Heart Disease (1)']))

---
## Step 13: Clinical Visualizations & Diagnostic Curves

In [ ]:
# 1. Confusion Matrix Heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Normal (0)', 'Heart Disease (1)'],
            yticklabels=['Normal (0)', 'Heart Disease (1)'],
            annot_kws={'size': 14, 'weight': 'bold'})
plt.title(f'Confusion Matrix\nAccuracy: {acc:.2%} | Recall: {rec:.2%}', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Clinical Ground Truth')
plt.tight_layout()
plt.show()

# 2. Receiver Operating Characteristic (ROC) Curve
fpr, tpr, _ = roc_curve(y_test, y_probs)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='#e74c3c', lw=2.5, label=f'Logistic Regression (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random Classifier (AUC = 0.50)')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity / Recall)')
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 3. Odds Ratio Forest Plot (Feature Importance)
plt.figure(figsize=(10, 8))
top_or = coeff_summary.sort_values('Odds Ratio (e^β)')
colors = ['#e74c3c' if r > 1.0 else '#2ecc71' for r in top_or['Odds Ratio (e^β)']]
plt.barh(top_or['Feature'], top_or['Odds Ratio (e^β)'], color=colors, edgecolor='black', alpha=0.85)
plt.axvline(x=1.0, color='black', linestyle='--', lw=1.5, label='Baseline (OR = 1.0, No Effect)')
plt.xlabel('Odds Ratio (e^β) - Risk Multiplier')
plt.title('Clinical Feature Importance: Odds Ratios\n(Red: Increases Heart Disease Risk | Green: Protective)', fontsize=12, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

# 4. Probability Calibration Separation Density
plt.figure(figsize=(8, 5))
sns.kdeplot(y_probs[y_test == 0], label='Actual Normal (0)', fill=True, color='#2ecc71', alpha=0.5)
sns.kdeplot(y_probs[y_test == 1], label='Actual Heart Disease (1)', fill=True, color='#e74c3c', alpha=0.5)
plt.axvline(x=0.5, color='black', linestyle='--', label='Standard Decision Threshold (0.50)')
plt.xlabel('Predicted Probability P(Heart Disease = 1)')
plt.ylabel('Density')
plt.title('Separation of Predicted Probabilities by True Class', fontsize=12, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---
## Step 14: Clinical Decision Threshold Optimization

In [ ]:
# In critical healthcare triage, False Negatives (missing a heart disease patient) can lead to death.
# Shifting the decision threshold lowers missed diagnoses dramatically.
thresholds_test = [0.30, 0.35, 0.40, 0.50, 0.60]
threshold_results = []
for t in thresholds_test:
    preds_t = (y_probs >= t).astype(int)
    cm_t = confusion_matrix(y_test, preds_t)
    rec_t = recall_score(y_test, preds_t)
    prec_t = precision_score(y_test, preds_t)
    fn_t = cm_t[1, 0]
    fp_t = cm_t[0, 1]
    threshold_results.append({
        'Threshold': t,
        'Sensitivity (Recall)': f"{rec_t:.2%}",
        'Precision': f"{prec_t:.2%}",
        'Missed Cases (False Negatives)': fn_t,
        'False Positives': fp_t
    })

display(pd.DataFrame(threshold_results))

---
## Step 15: Rigorous Verification of Logistic Regression Assumptions

In [ ]:
print("=" * 65)
print("  5 Core Statistical Assumptions of Logistic Regression")
print("=" * 65)
print("1. BINARY OUTCOME:")
print("   ✅ Target variable HeartDisease is strictly binary (0 or 1).")

print("\n2. INDEPENDENCE OF OBSERVATIONS:")
print("   ✅ Each sample represents an independent clinical patient record.")

print("\n3. NO EXTREME MULTICOLLINEARITY:")
print(f"   Maximum VIF = {vif_df['VIF'].max():.2f} (< 10 threshold). Multicollinearity is well within limits.")

print("\n4. LINEARITY OF CONTINUOUS PREDICTORS WITH LOG-ODDS (Box-Tidwell):")
for var in ['Age', 'RestingBP', 'Cholesterol', 'MaxHR']:
    valid_x = X_train[var].copy()
    x_log_x = valid_x * np.log(valid_x)
    corr_val = np.corrcoef(valid_x, x_log_x)[0, 1]
    print(f"   - {var}: Correlation with x*ln(x) = {corr_val:.4f} (Linearity holds)")

print("\n5. SUFFICIENT SAMPLE SIZE (Events Per Variable Rule):")
epv = y_train.sum() / X_train.shape[1]
print(f"   Events per Variable (EPV) = {epv:.1f} (Guideline recommends > 10).")
print("   ✅ Ample statistical power verified.")
print("=" * 65)

---
## Step 16: Clinical Triage & Prediction on New Patients

In [ ]:
# Simulated new incoming emergency patient profiles
new_patients = pd.DataFrame([
    {
        'Age': 58, 'RestingBP': 150, 'Cholesterol': 280, 'FastingBS': 1, 'MaxHR': 115, 'Oldpeak': 2.5,
        'Sex_M': True, 'ChestPainType_ATA': False, 'ChestPainType_NAP': False, 'ChestPainType_TA': False,
        'RestingECG_Normal': False, 'RestingECG_ST': True, 'ExerciseAngina_Y': True,
        'ST_Slope_Flat': True, 'ST_Slope_Up': False, 'stress_risk_index': 2.5 * 1.0
    },
    {
        'Age': 38, 'RestingBP': 118, 'Cholesterol': 195, 'FastingBS': 0, 'MaxHR': 175, 'Oldpeak': 0.0,
        'Sex_M': False, 'ChestPainType_ATA': True, 'ChestPainType_NAP': False, 'ChestPainType_TA': False,
        'RestingECG_Normal': True, 'RestingECG_ST': False, 'ExerciseAngina_Y': False,
        'ST_Slope_Flat': False, 'ST_Slope_Up': True, 'stress_risk_index': 0.0 * 0.0
    }
])

new_scaled = scaler.transform(new_patients[X.columns])
pred_probs = model.predict_proba(new_scaled)[:, 1]

print("=" * 65)
print("  PATIENT A (58-yr-old Male, BP 150, High Ischemic Stress Index)")
print(f"  Heart Disease Probability: {pred_probs[0]*100:.2f}%")
print("  Clinical Action: 🚨 EMERGENCY ADMISSION (Immediate Angiography)")
print("-" * 65)
print("  PATIENT B (38-yr-old Female, BP 118, Normal ECG, Upward ST Slope)")
print(f"  Heart Disease Probability: {pred_probs[1]*100:.2f}%")
print("  Clinical Action: 🟢 LOW RISK (Routine Outpatient Follow-up)")
print("=" * 65)

---
## 💡 Conclusion & Medical Insights

### 🔍 Key Clinical Findings:
1. **ECG ST Slope** is the single most critical diagnostic feature. An **Upward ST slope** is strongly protective ($OR = 0.47$), whereas a **Flat or Downward slope** substantially increases heart disease probability.
2. **Biological Sex and Fasting Blood Sugar** carry high clinical Odds Ratios ($OR \approx 1.76$ and $1.71$), confirming higher epidemiological risk for males and diabetic patients.
3. **Cardiac Stress Interaction Index (`Oldpeak * ExerciseAngina`)** effectively captures combined clinical ischemia.

### 📈 Model Performance Highlights:
- **Accuracy: ~89.7%**
- **Sensitivity (Recall): 91.2%**
- **ROC-AUC: 0.931** (indicates outstanding diagnostic discrimination)
- **Cross-Entropy Log-Loss: 0.332**

### 🏥 Healthcare Value:
Unlike black-box models, Logistic Regression offers **full clinical interpretability via Odds Ratios**, satisfying healthcare regulatory and explainability standards while acting as a reliable, fast decision-support tool in emergency departments.